# SmartSched: fine-tune the picker and compare

Runs the 4 test weeks with the base model, fine-tunes it with LoRA on the made-up students in `csp/train/`, runs the same 4 weeks again with the fine-tuned model, and prints a comparison table.

1. On your laptop, from the repo's `ml/` folder: `zip -r csp.zip eval csp -x '*/outputs/*' '*__pycache__*'`
2. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** and pick `csp.zip` when asked.

Takes about 1.5 hours if the training data is made here, about 1 hour if it's in the zip. Keep the tab open.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
WEEKS = ["01", "02", "03", "04"]

In [ ]:
# Pick csp.zip
from google.colab import files
files.upload()
!unzip -oq csp.zip && ls csp

In [ ]:
!pip install -q "transformers>=4.56" peft jsonschema
# Colab ships an old torchao that makes peft refuse to load. We don't use it.
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Training data: made here unless the zip already has it (about 30 min on Colab's CPU).
# Data from an older version (no fit_order) is made again, since eval_val.py needs it.
import json, os
def data_ok():
    try:
        return 'fit_order' in json.loads(open('csp/train/val.jsonl').readline())
    except (OSError, ValueError):
        return False
if not data_ok():
    !python csp/make_train.py --n 500 --val 40
assert data_ok(), 'training data missing: see the output above'
!wc -l csp/train/*.jsonl

## 1. Base model on the 4 weeks

In [ ]:
for w in WEEKS:
    !python csp/plan.py eval/prompts/{w}.json --model {MODEL} --device cuda --out runs/base/{w}

### Base model on 30 held-out students (about 5 min)

In [ ]:
!python csp/eval_val.py --n 30 --model {MODEL} --device cuda --out runs/val_base.json

## 2. Fine-tune (about 25–30 minutes)

In [ ]:
!python csp/finetune.py --model {MODEL} --out adapter
# Stop Run all here if training failed, instead of running the tuned model that doesn't exist.
assert os.path.exists('adapter/adapter_config.json'), 'training failed: see the error above'

## 3. Fine-tuned model on the same 4 weeks

Same seed, so it sees exactly the same calendars as the base run.

In [ ]:
for w in WEEKS:
    !python csp/plan.py eval/prompts/{w}.json --model {MODEL} --adapter adapter --device cuda --out runs/tuned/{w}

### Fine-tuned model on the same 30 held-out students

Phase 1 is done when this prints **MET**.

In [ ]:
!python csp/eval_val.py --n 30 --model {MODEL} --adapter adapter --device cuda --out runs/val_tuned.json

## 4. Compare

In [ ]:
!python csp/compare.py runs/base runs/tuned

In [ ]:
# Look at any report. Change the path to see another week or the base run.
from IPython.display import HTML
HTML(open("runs/tuned/04/report.html").read())

In [ ]:
# Download the runs and the adapter
!zip -qr results.zip runs adapter
files.download("results.zip")